In [0]:
# Afficher les colonnes customers
customers = spark.table("`E-commerce`.silver.customers")

print(customers.columns)

In [0]:
# Afficher les colonnes products
products = spark.table("`E-commerce`.silver.products")

print(products.columns)

In [0]:
# Afficher les colonnes transactions
transactions = spark.table("`E-commerce`.silver.transactions")

print(transactions.columns)

In [0]:
# Afficher les colonnes sessions
sessions = spark.table("`E-commerce`.silver.sessions")

print(sessions.columns)

In [0]:
# Afficher les colonnes reviews
reviews = spark.table("`E-commerce`.silver.reviews")

print(reviews.columns)

In [0]:
# Afficher quelques clients
display(customers.limit(10))

In [0]:
# Creer la dimension customers
dim_customers = customers.select(
    "customer_id",
    "signup_date",
    "age",
    "gender",
    "country",
    "segment",
    "is_churned",
    "lifetime_value"
)

In [0]:
# Enregistrer dim_customers dans Gold
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.dim_customers")

In [0]:
# Joindre les transactions avec les produits
sales_products = transactions.join(
    products,
    on="product_id",
    how="left"
)

In [0]:
# Comparer le nombre de lignes
print("Transactions :", transactions.count())
print("Apres jointure :", sales_products.count())

In [0]:
# Joindre avec les clients
sales_complete = sales_products.join(
    customers,
    on="customer_id",
    how="left"
)

In [0]:
# Comparer le nombre de lignes
print("Transactions :", transactions.count())
print("Apres les jointures :", sales_complete.count())

In [0]:
# Verifier les product_id sans correspondance
products_ids = products.select("product_id")

missing_products = transactions.join(
    products_ids,
    on="product_id",
    how="left_anti"
).count()

print("Produits sans correspondance :", missing_products)

In [0]:
# Verifier les customer_id sans correspondance
customers_ids = customers.select("customer_id")

missing_customers = transactions.join(
    customers_ids,
    on="customer_id",
    how="left_anti"
).count()

print("Clients sans correspondance :", missing_customers)

In [0]:
# Creer la table fact_sales
fact_sales = sales_complete.select(
    "transaction_id",
    "transaction_date",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "total_amount"
)

In [0]:
# Enregistrer fact_sales dans Gold
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.fact_sales")

In [0]:
# Creer fact_sessions
fact_sessions = sessions.select(
    "session_id",
    "customer_id",
    "session_date",
    "device",
    "channel",
    "duration_seconds",
    "pages_viewed",
    "converted",
    "bounced",
    "cart_additions"
)

In [0]:
# Enregistrer fact_sessions dans Gold
fact_sessions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.fact_sessions")

In [0]:
# Verifier la relation sessions-clients
missing_session_customers = fact_sessions.join(
    dim_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).count()

print("Clients sans correspondance :", missing_session_customers)

In [0]:
# Creer fact_reviews
fact_reviews = reviews.select(
    "review_id",
    "customer_id",
    "product_id",
    "review_date",
    "rating",
    "review_text",
    "helpful_votes",
    "verified_purchase"
)

In [0]:
# Verifier les clients des reviews
missing_review_customers = fact_reviews.join(
    dim_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).count()

print("Clients sans correspondance :", missing_review_customers)

In [0]:
# Enregistrer fact_reviews dans Gold
fact_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.fact_reviews")

In [0]:
# Afficher les tables Gold
spark.sql("SHOW TABLES IN `E-commerce`.gold").show(truncate=False)

In [0]:
# Creer la dimension produits
dim_products = products.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "price",
    "stock_quantity",
    "discount_pct",
    "is_featured",
    "weight_kg"
)

In [0]:
# Enregistrer dim_products dans Gold
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.dim_products")

####Objectif 1 — Monitor Sales Performance
####KPI 1 : Total Revenue

####Question métier : « Combien d'argent les ventes ont-elles généré au total ? »

In [0]:
# Charger les ventes Gold
fact_sales = spark.table("`E-commerce`.gold.fact_sales")

In [0]:
# Calculer le revenu total

from pyspark.sql.functions import sum
total_revenue = fact_sales.agg(

    sum("total_amount").alias("total_revenue")
)

display(total_revenue)

In [0]:
# Compter les transactions
number_transactions = fact_sales.select(
    "transaction_id"
).distinct().count()

print("Number of Transactions :", number_transactions)

In [0]:
# Calculer le montant moyen des transactions
from pyspark.sql.functions import avg

average_order_value = fact_sales.agg(
    avg("total_amount").alias("average_order_value")
)

display(average_order_value)

In [0]:
# Calculer le revenu par mois
from pyspark.sql.functions import date_format, sum

sales_trend = fact_sales \
    .withColumn("month", date_format("transaction_date", "yyyy-MM")) \
    .groupBy("month") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("month")

display(sales_trend)

####Objectif métier 2 — Analyze Product Performance

####La question métier est :

####Quels produits et quelles catégories génèrent le plus de ventes et de revenus ?

In [0]:
# Charger les produits Gold
dim_products = spark.table("`E-commerce`.gold.dim_products")

In [0]:
# Joindre les ventes avec les produits
sales_products = fact_sales.join(
    dim_products,
    on="product_id",
    how="left"
)

In [0]:
# Calculer le revenu par produit
revenue_by_product = sales_products \
    .groupBy("product_id", "product_name") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("revenue", ascending=False)

display(revenue_by_product)

In [0]:
# Calculer les quantites vendues par produit
best_selling_products = sales_products \
    .groupBy("product_id", "product_name") \
    .agg(sum("quantity").alias("quantity_sold")) \
    .orderBy("quantity_sold", ascending=False)

display(best_selling_products)

In [0]:
# Calculer le revenu par categorie
revenue_by_category = sales_products \
    .groupBy("category") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("revenue", ascending=False)

display(revenue_by_category)

####Objectif 3 — Understand Customer Behavior

In [0]:
# Charger les tables Gold
dim_customers = spark.table("`E-commerce`.gold.dim_customers")
fact_sales = spark.table("`E-commerce`.gold.fact_sales")
fact_sessions = spark.table("`E-commerce`.gold.fact_sessions")

In [0]:
# Joindre les ventes avec les clients
sales_customers = fact_sales.join(
    dim_customers,
    on="customer_id",
    how="left"
)

In [0]:
# Calculer le revenu par segment
revenue_by_segment = sales_customers \
    .groupBy("segment") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("revenue", ascending=False)

display(revenue_by_segment)

In [0]:
# Calculer le revenu par pays
revenue_by_country = sales_customers \
    .groupBy("country") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("revenue", ascending=False)

display(revenue_by_country)

In [0]:
# Calculer le comportement moyen des clients
from pyspark.sql.functions import avg
customer_activity = fact_sessions \
    .groupBy("customer_id") \
    .agg(
        avg("pages_viewed").alias("avg_pages_viewed"),
        avg("duration_seconds").alias("avg_session_duration"),
        avg("cart_additions").alias("avg_cart_additions")
    )

display(customer_activity)

In [0]:
# Calculer le comportement moyen des clients
customer_activity = fact_sessions \
    .groupBy("customer_id") \
    .agg(
        avg("pages_viewed").alias("avg_pages_viewed"),
        avg("duration_seconds").alias("avg_session_duration"),
        avg("cart_additions").alias("avg_cart_additions")
    )

display(customer_activity)

In [0]:
# Calculer le taux de conversion
conversion_rate = fact_sessions \
    .agg(
        (avg("converted") * 100).alias("conversion_rate")
    )

display(conversion_rate)

In [0]:
# Calculer le taux de rebond
bounce_rate = fact_sessions \
    .agg(
        (avg("bounced") * 100).alias("bounce_rate")
    )

display(bounce_rate)

In [0]:
# Calculer le taux de conversion par canal
conversion_by_channel = fact_sessions \
    .groupBy("channel") \
    .agg(
        (avg("converted") * 100).alias("conversion_rate")
    ) \
    .orderBy("conversion_rate", ascending=False)

display(conversion_by_channel)

#####Objectif métier 6 — Analyze Marketing Channels

In [0]:
# Calculer la performance de chaque canal
from pyspark.sql.functions import count, sum, avg

channel_performance = fact_sessions \
    .groupBy("channel") \
    .agg(
        count("session_id").alias("total_sessions"),
        sum("converted").alias("total_conversions"),
        (avg("converted") * 100).alias("conversion_rate")
    ) \
    .orderBy("total_conversions", ascending=False)

display(channel_performance)

#####Objectif métier 7 — Analyze Device Performance

In [0]:
# Calculer la performance par appareil
device_performance = fact_sessions \
    .groupBy("device") \
    .agg(
        count("session_id").alias("total_sessions"),
        sum("converted").alias("total_conversions"),
        (avg("converted") * 100).alias("conversion_rate")
    ) \
    .orderBy("conversion_rate", ascending=False)

display(device_performance)

In [0]:
# Calculer le taux de rebond par appareil
bounce_by_device = fact_sessions \
    .groupBy("device") \
    .agg(
        (avg("bounced") * 100).alias("bounce_rate")
    ) \
    .orderBy("bounce_rate", ascending=False)

display(bounce_by_device)

In [0]:
# Charger les reviews Gold
fact_reviews = spark.table("`E-commerce`.gold.fact_reviews")

In [0]:
# Calculer la note moyenne
average_rating = fact_reviews \
    .agg(avg("rating").alias("average_rating"))

display(average_rating)

In [0]:
# Compter les reviews par note
reviews_by_rating = fact_reviews \
    .groupBy("rating") \
    .agg(count("review_id").alias("number_reviews")) \
    .orderBy("rating")

display(reviews_by_rating)

In [0]:
# Joindre reviews et produits
reviews_products = fact_reviews.join(
    dim_products,
    on="product_id",
    how="left"
)

In [0]:
# Calculer la note moyenne par produit
rating_by_product = reviews_products \
    .groupBy("product_id", "product_name") \
    .agg(
        avg("rating").alias("average_rating"),
        count("review_id").alias("number_reviews")
    ) \
    .orderBy("average_rating", ascending=False)

display(rating_by_product)

In [0]:
# Afficher les produits les moins bien notes
poorly_rated_products = rating_by_product \
    .orderBy("average_rating", ascending=True)

display(poorly_rated_products)

In [0]:
# Enregistrer les ventes mensuelles
sales_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.sales_trend")

In [0]:
# Enregistrer le revenu par produit
revenue_by_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.product_performance")

In [0]:
# Enregistrer le revenu par categorie
revenue_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.category_performance")

In [0]:
# Enregistrer le revenu par segment client
revenue_by_segment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.customer_segment_performance")

In [0]:
# Enregistrer la performance des canaux
channel_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.channel_performance")

In [0]:
# Enregistrer la performance des appareils
device_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.device_performance")

In [0]:
# Enregistrer la satisfaction par produit
rating_by_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.gold.product_satisfaction")

In [0]:
# Afficher toutes les tables Gold
spark.sql("SHOW TABLES IN `E-commerce`.gold").show(truncate=False)